# HybridPdM v2.1 - Colab Runner

**목적**: 로컬에서 리소스 부족으로 실행하지 못한 N-CMAPSS LSTM을 GPU 환경에서 실행.

**사전 준비**:
1. Google Drive에 `hybrid_pdm` 폴더를 통째로 업로드 (dataset 포함)
2. 런타임 유형: **GPU (T4)** 선택
3. 셀을 순서대로 실행

## 1. Google Drive 마운트 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === 경로 설정 ===
# Google Drive 내 hybrid_pdm 폴더 경로를 본인 환경에 맞게 수정하세요.
PROJECT_DIR = '/content/drive/MyDrive/hybrid_pdm'

import os
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

## 2. 의존성 설치 + GPU 확인

In [ ]:
!pip install -q torch numpy pandas scikit-learn scipy h5py captum

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 3. 전체 파이프라인 실행 (N-CMAPSS 포함)

로컬에서 달성한 결과를 GPU 환경에서 재현 + N-CMAPSS 추가 실행.

In [ ]:
# 전체 파이프라인 (모든 데이터셋)
!python main.py --skip-explain

## 4. (선택) N-CMAPSS 강화 실행

위 기본 실행 결과가 좋지 않을 경우, 아래 셀에서 하이퍼파라미터를 강화하여 재시도.

**변경점**:
- hidden: 128 -> 256 (GPU VRAM 활용)
- epochs: 80 -> 150 (더 충분한 학습, early stopping이 보호)
- window: 30 -> 50 (더 긴 시간적 맥락)
- lr: 1e-3 -> 5e-4 (큰 모델에 맞는 낮은 학습률)

In [ ]:
import config
import importlib

# config 모듈 리로드 (이전 셀에서 import 되었을 수 있음)
importlib.reload(config)

# N-CMAPSS 강화 설정 적용
config.LSTM_CFG.update({
    "hidden": 256,           # 128 -> 256
    "epochs": 150,           # 80 -> 150
    "window": 50,            # 30 -> 50
    "lr": 5e-4,              # 1e-3 -> 5e-4
    "early_stop_patience": 15,  # 10 -> 15
})

print('=== N-CMAPSS 강화 설정 ===')
for k, v in config.LSTM_CFG.items():
    print(f'  {k}: {v}')

In [ ]:
# N-CMAPSS만 강화 설정으로 재실행
!python main.py --datasets ncmapss_lstm --skip-explain

## 5. (선택) C-MAPSS도 강화 실행

C-MAPSS는 이미 r²=0.885로 목표 달성이지만, GPU 환경에서 더 높은 성능을 노릴 수 있음.

In [ ]:
# C-MAPSS + N-CMAPSS 동시 실행 (강화 설정 적용 상태)
!python main.py --datasets cmapss_lstm ncmapss_lstm --skip-explain

## 6. 결과 리포트 확인

In [ ]:
import json
from pathlib import Path

report_dir = Path('artifacts/reports')
reports = sorted(report_dir.glob('pipeline_report_*.json'), reverse=True)

if reports:
    latest = reports[0]
    print(f'Latest report: {latest.name}\n')
    with open(latest) as f:
        data = json.load(f)
    for entry in data:
        name = entry.get('name', '?')
        status = entry.get('status', '?')
        ev = entry.get('eval', {})
        keys = ['accuracy', 'f1', 'test_f1', 'rmse', 'mae', 'r2',
                'best_threshold', 'best_percentile', 'decision_threshold']
        summary = {k: round(v, 4) if isinstance(v, float) else v
                   for k, v in ev.items() if k in keys}
        print(f'{name:20s}  [{status:5s}]  {summary}')
else:
    print('No reports found.')